In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog qiskit-serverless
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Isulat ang iyong unang Qiskit Serverless na programa

<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa page na ito ay ginawa gamit ang mga sumusunod na requirements.
Inirerekomenda naming gamitin ang mga bersyong ito o mas bago.

```
qiskit[all]~=1.3.1
qiskit-ibm-runtime~=0.34.0
qiskit-aer~=0.15.1
qiskit-serverless~=0.18.1
qiskit-ibm-catalog~=0.2
qiskit-addon-sqd~=0.8.1
qiskit-addon-utils~=0.1.0
qiskit-addon-mpf~=0.2.0
qiskit-addon-aqc-tensor~=0.1.2
qiskit-addon-obp~=0.1.0
scipy~=1.15.0
pyscf~=2.8.0
```
</details>
Ang halimbawang ito ay nagpapakita kung paano gamitin ang mga tool ng `qiskit-serverless` para gumawa ng parallel transpilation na programa, at pagkatapos ay gamitin ang `qiskit-ibm-catalog` para i-deploy ang iyong programa sa IBM Quantum Platform bilang reusable remote service.

## Halimbawa: remote transpilation gamit ang Qiskit Serverless
Simulan sa sumusunod na halimbawa na nagta-transpile ng isang `circuit` laban sa isang ibinigay na `backend` at target na `optimization_level`, at unti-unting dagdagan ng mas maraming elemento para i-deploy ang iyong workload sa Qiskit Serverless.

Ilagay ang sumusunod na code cell sa file na `./source_files/transpile_remote.py`. Ang file na ito ang programa na ia-upload sa Qiskit Serverless.

In [ ]:
# This cell is hidden from users, it creates a new folder
from pathlib import Path

Path("./source_files").mkdir(exist_ok=True)

In [ ]:
%%writefile ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this locally in a notebook), running this cell saves to disk rather than executing the code.

from qiskit.transpiler import generate_preset_pass_manager

def transpile_remote(circuit, optimization_level, backend):
    """Transpiles an abstract circuit into an ISA circuit for a given backend."""
    pass_manager = generate_preset_pass_manager(
        optimization_level=optimization_level,
		backend=backend
    )
    isa_circuit = pass_manager.run(circuit)
    return isa_circuit

### Add code to get program arguments

Now add the following code to your program file, which sets up program arguments.

Your initial `transpile_remote.py` has three inputs: `circuits`, `backend_name`, and `optimization_level`. Serverless is currently limited to only accept serializable inputs and outputs. For this reason, you cannot pass in `backend` directly, so use `backend_name` as a string instead.

In [ ]:
%%writefile --append ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this locally in a notebook), running this cell saves to disk rather than executing the code.

from qiskit_serverless import get_arguments, save_result, distribute_task, get

# Get program arguments
arguments = get_arguments()
circuits = arguments.get("circuits")
backend_name = arguments.get("backend_name")
optimization_level = arguments.get("optimization_level")

## I-set up ang iyong mga file
Kailangan ng Qiskit Serverless na i-set up ang mga `.py` file ng iyong workload sa isang dedicated na direktoryo. Ang sumusunod na istruktura ay halimbawa ng magandang gawi:

In [ ]:
%%writefile --append ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this locally in a notebook), running this cell saves to disk rather than executing the code.

from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.backend(backend_name)

Ina-upload ng Serverless ang mga nilalaman ng `source_files` para patakbuhin nang remote. Kapag na-set up na ito, maaari mong baguhin ang `transpile_remote.py` para kumuha ng mga input at magbalik ng mga output.

### Kunin ang mga argumento ng programa
Ang iyong paunang `transpile_remote.py` ay may tatlong input: `circuits`, `backend_name`, at `optimization_level`. Limitado pa lang ang Serverless na tumatanggap lamang ng serializable na mga input at output. Dahil dito, hindi mo maaaring direktang ipasa ang `backend`, kaya gamitin ang `backend_name` bilang string sa halip.

In [ ]:
%%writefile --append ./source_files/transpile_remote.py
# If you include the preceding `%%writefile` command (visible only when you read this locally in a notebook), running this cell saves to disk rather than executing the code.

# Each circuit is being transpiled and will populate the array
results = [
    transpile_remote(circuit, 1, backend)
    for circuit in circuits
]

save_result({
    "transpiled_circuits": results
})

<span id="upload-to-qiskit-serverless"></span>
### Authenticate to Qiskit Serverless

Use `qiskit-ibm-catalog` to authenticate to `QiskitServerless` with your API key (you can use your `QiskitRuntimeService` API key, or create a new API key on the [IBM Quantum Platform dashboard](https://quantum.cloud.ibm.com)).

In [13]:
from qiskit_ibm_catalog import QiskitServerless, QiskitFunction

# Authenticate to the remote cluster and submit the pattern for remote execution
serverless = QiskitServerless()

Sa puntong ito, maaari mong makuha ang iyong backend gamit ang `QiskitRuntimeService` at idagdag ang iyong umiiral na programa gamit ang sumusunod na code. Kailangan na [na-save mo na ang iyong credentials](/guides/cloud-setup) para sa sumusunod na code.

In [8]:
transpile_remote_demo = QiskitFunction(
    title="transpile_remote_serverless",
    entrypoint="transpile_remote.py",
    working_dir="./source_files/",
)

In [9]:
serverless.upload(transpile_remote_demo)

QiskitFunction(transpile_remote_serverless)

Sa wakas, maaari mong patakbuhin ang `transpile_remote()` sa lahat ng `circuits` na ipinasa, at ibalik ang `transpiled_circuits` bilang resulta:

In [10]:
# Get program from serverless.list() that matches the title of the one we uploaded
next(
    program
    for program in serverless.list()
    if program.title == "transpile_remote_serverless"
)

QiskitFunction(transpile_remote_serverless)

In [11]:
# This cell is hidden from users, it checks the program uploaded correctly
assert _.title == "transpile_remote_serverless"  # noqa: F821

In [12]:
# This cell is hidden from users, it checks the program executes correctly
from time import sleep
from qiskit import QuantumCircuit

qc = QuantumCircuit(2)
transpile_remote_serverless = serverless.load("transpile_remote_serverless")
job = transpile_remote_serverless.run(
    circuits=[qc],
    backend="ibm_sherbrooke",
    optimization_level=1,
)
while True:
    sleep(5)
    status = job.status()
    if status not in ["QUEUED", "INITIALIZING", "RUNNING", "DONE"]:
        raise Exception(
            f"Unexpected job status: '{status}'\n"
            + "Here are the logs:\n"
            + job.logs()
        )
    if status == "DONE":
        break

Kino-compress ng Qiskit Serverless ang mga nilalaman ng `working_dir` (sa kasong ito, `source_files`) sa isang `tar`, na ini-upload at nilinis pagkatapos. Tinutukoy ng `entrypoint` ang pangunahing executable na programa para patakbuhin ng Qiskit Serverless. Bukod dito, kung ang iyong programa ay may custom na `pip` na dependencies, maaari mo itong idagdag sa isang `dependencies` array: